# 📍 Assistente Inteligente de Rotas e Campo (Field Management)
 **Projeto de Otimização Geoespacial e Automação de Visitas Médicas**

Este notebook apresenta uma solução completa para automatizar, organizar e otimizar a rotina de visitas de um representante farmacêutico em Manaus. A ferramenta integra ingestão resiliente de dados, cálculo de prioridades de contacto, visualização em mapa interativo, atalhos de comunicação via WhatsApp e briefings diários narrados por Inteligência Artificial.

**Tecnologias Utilizadas:** `Python`, `Pandas`, `Folium`, `Geopy`, `gTTS (Google Text-to-Speech)`.

## 🛠️ 1. Configuração do Ambiente e Importação de Dependências

Nesta etapa, carregamos todas as bibliotecas necessárias para manipulação de dados, geocodificação, renderização de mapas interativos e síntese de áudio.

## 📥 2. Ingestão Inteligente e Resiliente de Dados

Este bloco é responsável por importar os dados diretamente do Google Sheets com deteção automática de cabeçalho e mecanismo de segurança (*fallback*) em caso de falha de conexão.

## 📊 3. Regras de Negócio e Priorização de Visitas

Aplicação da lógica de cadência para categorizar o nível de urgência de cada contacto (Reativação Urgente, Atenção ou Em Dia) com base no histórico de visitas anteriores.

## 🗺️ 4. Mapeamento Geoespacial Interativo (Manaus)

Tratamento de endereços e plotagem no mapa de Manaus. Cada marcador inclui informação do médico, grau de prioridade e links diretos para navegação (Waze / Google Maps).

## 🎙️ 5. Geração do Briefing Diário em Áudio (IA Voice)

Sintetização automática do texto da rota num ficheiro de áudio `.mp3` para que o propagandista ouça o resumo das prioridades do dia no carro ou no telemóvel.

In [ ]:
# 1. URL da planilha do Google Sheets (exportação em CSV)
url_planilha = "https://docs.google.com/spreadsheets/d/SEU_ID_DA_PLANILHA/gviz/tq?tqx=out:csv"

try:
    # Lê a planilha bruta sem definir cabeçalho fixo
    df_raw = pd.read_csv(url_planilha, header=None)

    # Encontra automaticamente a linha exata onde aparece a coluna "CRM"
    linha_cabecalho = df_raw[df_raw.isin(['CRM']).any(axis=1)].index[0]

    # Recarrega a planilha definindo essa linha encontrada como o cabeçalho oficial
    df_visitas = pd.read_csv(url_planilha, skiprows=linha_cabecalho)

    # Limpa espaços em branco nos nomes das colunas
    df_visitas.columns = df_visitas.columns.str.strip()

    # Remove linhas totalmente vazias, caso existam
    df_visitas = df_visitas.dropna(subset=['CRM'])

    print("✅ Planilha carregada e alinhada com sucesso!")

except Exception as e:
    print(f"⚠️ A usar dados locais de segurança devido a: {e}")
    # Dados de segurança (Fallback) caso ocorra qualquer interrupção de rede
    dados_fallback = {
        'CRM': ['12345-AM', '67890-AM'],
        'Nome_Medico': ['Dr. Carlos Silva', 'Dra. Ana Souza'],
        'Especialidade': ['Dermatologia', 'Pediatria'],
        'Target': ['Alto (A)', 'Médio (B)'],
        'Perfil_Comportamental': ['Analítico / Científico', 'Prático / Agilizado'],
        'Horario_Preferencial': ['Segundas (08h - 10h)', 'Quartas (14h - 16h)'],
        'Linha_Produtos': ['Produto A / Produto B', 'Produto C / Produto J'],
        'Resumo_Ultima_Visita': ['Pediu estudo comparativo.', 'Boa aceitação.'],
        'Proxima_Acao': ['Entregar artigo científico.', 'Levar amostras.']
    }
    df_visitas = pd.DataFrame(dados_fallback)

In [ ]:
# 2. Função para gerar a ficha de preparação do médico
def gerar_ficha_preparacao(termo_busca):
    # Procura pelo CRM ou pelo Nome do Médico usando os nomes reais das colunas
    resultado = df_visitas[
        (df_visitas['CRM'].astype(str).str.contains(termo_busca, case=False, na=False)) |
        (df_visitas['Nome do Médico'].astype(str).str.contains(termo_busca, case=False, na=False))
    ]

    if not resultado.empty:
        for _, info in resultado.iterrows():
            print("=" * 55)
            print(f"🏥 PREPARAÇÃO DE VISITA - LABORATÓRIO EXEMPLO")
            print("=" * 55)
            print(f"👨‍⚕️ Médico: {info['Nome do Médico']} | CRM: {info['CRM']}")
            print(f"🩺 Especialidade: {info['Especialidade']} | Target: {info['Target']}")
            print(f"🧠 Perfil: {info['Perfil Comportamental']}")
            print(f"⏰ Horário Ideal: {info['Horário Preferencial']}")
            print(f"💊 Foco de Produto: {info['Linha de Produtos']}")
            print("-" * 55)
            print(f"📝 RESUMO DA ÚLTIMA VISITA:")
            print(f"   \"{info['Resumo da Última Visita']}\"")
            print("-" * 55)
            print(f"🎯 COMPROMISSO PARA HOJE:")
            print(f"   👉 {info['Próxima Ação']}")
            print("=" * 55)
            print("\n")
    else:
        print(f"⚠️ Nenhum médico localizado com o termo '{termo_busca}'.")

# Exemplo de teste: Buscar pelo nome do médico
gerar_ficha_preparacao('Carlos')

In [ ]:
# Listar os nomes das colunas atuais do DataFrame
list(df_visitas.columns)

In [ ]:
# Exibir estatísticas descritivas das colunas do DataFrame
display(df_visitas.describe(include='all'))

In [ ]:
# Filtrar médicos com Target 'Alto (A)'
medicos_alto_target = df_visitas[df_visitas['Target'] == 'Alto (A)']

# Exibir o DataFrame filtrado
display(medicos_alto_target[['CRM', 'Nome do Médico', 'Especialidade', 'Target']])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Definir o estilo do gráfico
sns.set_theme(style="whitegrid")
plt.figure(figsize=(8, 5))

# Criar o gráfico de contagem de médicos por especialidade
ax = sns.countplot(
    data=df_visitas,
    x='Especialidade',
    hue='Especialidade',
    palette='viridis',
    legend=False
)

# Customizações estéticas
plt.title('Contagem de Médicos por Especialidade', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Especialidade', fontsize=12)
plt.ylabel('Quantidade de Médicos', fontsize=12)
plt.yticks(range(0, df_visitas['Especialidade'].value_counts().max() + 2))

# Adicionar os valores acima de cada barra
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='baseline',
                fontsize=11, color='black',
                xytext=(0, 5),
                textcoords='offset points')

plt.tight_layout()
plt.show()

In [ ]:
# Calcular a frequência de cada Perfil Comportamental
frequencia_perfis = df_visitas['Perfil Comportamental'].value_counts().reset_index()
frequencia_perfis.columns = ['Perfil Comportamental', 'Quantidade']

# Exibir o resultado
display(frequencia_perfis)

In [ ]:
def filtrar_por_especialidade(especialidade):
    # Filtrar o DataFrame de forma case-insensitive (ignorando maiúsculas/minúsculas)
    resultado = df_visitas[
        df_visitas['Especialidade'].astype(str).str.contains(especialidade, case=False, na=False)
    ]

    if not resultado.empty:
        print(f"📌 Médicos encontrados para a especialidade '{especialidade}':")
        display(resultado[['CRM', 'Nome do Médico', 'Especialidade', 'Target', 'Bairro']])
    else:
        print(f"⚠️ Nenhum médico encontrado para a especialidade '{especialidade}'.")

# Exemplo de teste da função
filtrar_por_especialidade('Dermatologia')

In [ ]:
import pandas as pd
from datetime import datetime

# 1. Renomear a coluna 'Unnamed: 11' para refletir que se trata da 'Data da Última Visita'
if 'Unnamed: 11' in df_visitas.columns:
    df_visitas = df_visitas.rename(columns={'Unnamed: 11': 'Data da Última Visita'})

# 2. Converter a coluna para formato datetime de forma segura (tratando dia/mês/ano)
df_visitas['Data da Última Visita'] = pd.to_datetime(df_visitas['Data da Última Visita'], format='%d/%m/%Y', errors='coerce')

# 3. Calcular a diferença em dias com relação à data de hoje
data_hoje = pd.to_datetime(datetime.now().date())
df_visitas['Dias Desde a Última Visita'] = (data_hoje - df_visitas['Data da Última Visita']).dt.days

# Exibir os resultados para conferência
display(df_visitas[['CRM', 'Nome do Médico', 'Data da Última Visita', 'Dias Desde a Última Visita']])

In [ ]:
# Filtrar e listar os médicos com suas respectivas ações planejadas para hoje
ações_hoje = df_visitas[['CRM', 'Nome do Médico', 'Próxima Ação']].dropna(subset=['Próxima Ação'])

print("🎯 AÇÕES PLANEJADAS PARA HOJE:")
display(ações_hoje)

In [ ]:
# Exportar a lista de ações de hoje para um arquivo CSV
nome_arquivo = 'acoes_hoje.csv'
ações_hoje.to_csv(nome_arquivo, index=False, encoding='utf-8-sig')

print(f"✅ Arquivo '{nome_arquivo}' gerado com sucesso e pronto para download!")

In [ ]:
# Criar uma função correta para o status de retorno baseada nos dias da última visita
def definir_status_retorno(dias):
    if dias > 7:
        return "🔴 Urgente: Reagendar Visita"
    elif dias >= 5:
        return "🟡 Atenção: Programar Retorno"
    else:
        return "🟢 Em Dia"

# Aplicar a função à coluna correta
df_visitas['Status_Retorno'] = df_visitas['Dias Desde a Última Visita'].apply(definir_status_retorno)

# Exibir a tabela completa com o status de prioridade organizada (corrigido 'Nome do Médico')
display(df_visitas[['CRM', 'Nome do Médico', 'Dias Desde a Última Visita', 'Status_Retorno']])

In [ ]:
# Instalar a biblioteca de geocodificação gratuita (caso ainda não tenha)
!pip install -q geopy

from geopy.geocoders import Nominatim
import time

# Inicializar o geocodificador Nominatim (OpenStreetMap) com um identificador único
geolocator = Nominatim(user_agent="visit_optimizer_portfolio_am")

print("📍 A iniciar o mapeamento dos consultórios em Manaus...")

# Função para obter latitude e longitude baseada no endereço e bairro
def obter_coordenadas(endereco, bairro):
    query = f"{endereco}, {bairro}, Manaus, AM, Brasil"
    try:
        location = geolocator.geocode(query, timeout=10)
        time.sleep(1) # Pausa curta exigida pela política de uso gratuito da API
        if location:
            return location.latitude, location.longitude
        else:
            return None, None
    except Exception as e:
        print(f"Erro ao geocodificar {query}: {e}")
        return None, None

# Aplicar a geocodificação à nossa base de dados (se as colunas existirem)
latitudes = []
longitudes = []

for _, row in df_visitas.iterrows():
    end = row.get('Endereço da Clínica', 'Av. Djalma Batista')
    bairro = row.get('Bairro', 'Manaus')
    lat, lon = obter_coordenadas(end, bairro)
    latitudes.append(lat)
    longitudes.append(lon)

df_visitas['Latitude'] = latitudes
df_visitas['Longitude'] = longitudes

# Exibir os médicos com suas respetivas coordenadas geográficas (corrigido 'Nome do Médico')
display(df_visitas[['CRM', 'Nome do Médico', 'Bairro', 'Latitude', 'Longitude', 'Status_Retorno']])

In [ ]:
import folium

# 1. Criar o mapa centrado em Manaus usando o provedor Esri World Street Map (totalmente livre de bloqueios)
mapa_visitas = folium.Map(
    location=[-3.107914, -60.011643],
    zoom_start=13,
    tiles="Esri.WorldStreetMap"
)

# Cores dos marcadores consoante a urgência
cores_status = {
    "🔴 Urgente: Reagendar Visita": "red",
    "🟡 Atenção: Programar Retorno": "orange",
    "🟢 Em Dia": "green"
}

In [ ]:
# 2. Adicionar cada consultório como um pino no mapa
for _, row in df_visitas.iterrows():
    if pd.notna(row['Latitude']) and pd.notna(row['Longitude']):
        cor = "blue"
        for st, c in cores_status.items():
            if st in str(row.get('Status_Retorno', '')):
                cor = c
                break

        popup_html = f"""
        <div style="font-family: Arial; font-size: 12px; width: 180px;">
            <b>{row['Nome do Médico']}</b><br>
            <b>Especialidade:</b> {row['Especialidade']}<br>
            <b>Bairro:</b> {row['Bairro']}<br>
            <b>Status:</b> {row.get('Status_Retorno', 'N/A')}
        </div>
        """

        folium.Marker(
            location=[row['Latitude'], row['Longitude']],
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=f"{row['Nome do Médico']} ({row['Bairro']})",
            icon=folium.Icon(color=cor, icon="info-sign")
        ).add_to(mapa_visitas)

# Exibir o mapa interativo limpo no Colab
mapa_visitas

In [ ]:
import urllib.parse

# Função para gerar os links de navegação GPS para cada médico
def gerar_links_navegacao(df):
    relatorio_visitas = []

    print("=" * 65)
    print("📋 CRONOGRAMA DE VISITAS RECOMENDADO - LABORATÓRIO EXEMPLO")
    print("=" * 65)

    for _, row in df.iterrows():
        # Obter dados com segurança
        medico = row.get('Nome do Médico', 'Médico')
        bairro = row.get('Bairro', 'Manaus')
        status = row.get('Status_Retorno', 'Em Dia')
        lat = row.get('Latitude')
        lon = row.get('Longitude')

        # Gerar link do Google Maps/Waze baseado nas coordenadas ou endereço
        if pd.notna(lat) and pd.notna(lon):
            link_gps = f"https://www.google.com/maps/search/?api=1&query={lat},{lon}"
        else:
            query_endereco = urllib.parse.quote(f"{medico}, {bairro}, Manaus, AM")
            link_gps = f"https://www.google.com/maps/search/?api=1&query={query_endereco}"

        print(f"👨‍⚕️ {medico} ({bairro})")
        print(f"   📊 Status: {status}")
        print(f"   🗺️ Abrir no GPS: {link_gps}")
        print("-" * 65)

# Executar a geração do relatório para o teste
gerar_links_navegacao(df_visitas)

In [ ]:
import urllib.parse

# 1. Definir o número do destinatário (Exemplo: 55929XXXXXXXX)
numero_whatsapp = "55929XXXXXXXX"  # Preencha com o número dele com DDD

In [ ]:
# 2. Montar a mensagem formatada para o WhatsApp
mensagem_wa = "📋 *ROTA DE VISITAS RECOMENDADA - LABORATÓRIO EXEMPLO*\n\n"

for _, row in df_visitas.iterrows():
    medico = row.get('Nome do Médico', 'Médico')
    bairro = row.get('Bairro', 'Manaus')
    status = row.get('Status_Retorno', 'Em Dia')
    lat = row.get('Latitude')
    lon = row.get('Longitude')

    link_gps = f"https://www.google.com/maps/search/?api=1&query={lat},{lon}"

    mensagem_wa += f"👨‍⚕️ *{medico}* ({bairro})\n"
    mensagem_wa += f"📊 Status: {status}\n"
    mensagem_wa += f"🗺️ GPS: {link_gps}\n"
    mensagem_wa += "-----------------------------------\n"

In [ ]:
# 3. Converter a mensagem em formato URL seguro
mensagem_codificada = urllib.parse.quote(mensagem_wa)
link_envio_whatsapp = f"https://api.whatsapp.com/send?phone={numero_whatsapp}&text={mensagem_codificada}"

In [ ]:
# 4. Exibir o botão / link para enviar ao vivo
print("✅ Relatório formatado para o WhatsApp com sucesso!\n")
print("Clique no link abaixo para testar o envio direto para o celular dele ao vivo:")
print(link_envio_whatsapp)

In [ ]:
from gtts import gTTS
import os

# 1. Criar o texto do áudio de forma limpa e sem duplicar
texto_audio = "Bom dia. Segue o resumo da tua rota de visitas para hoje em Manaus. "
texto_audio += "Tens visitas prioritárias para reativar o contacto. "

for _, row in df_visitas.iterrows():
    medico_bruto = row.get('Nome do Médico', 'Médico')
    bairro = row.get('Bairro', 'Manaus')
    status = row.get('Status_Retorno', 'Em Dia')

    # Identifica se é feminino ou masculino com base no prefixo original e limpa o nome
    if "Dra" in medico_bruto or "Dra." in medico_bruto:
        tratamento = "Doutora"
        medico_limpo = medico_bruto.replace("Dra. ", "").replace("Dra ", "")
    else:
        tratamento = "Doutor"
        medico_limpo = medico_bruto.replace("Dr. ", "").replace("Dr ", "")

    if "Urgente" in status or "Atenção" in status:
        texto_audio += f"A {tratamento} {medico_limpo}" if tratamento == "Doutora" else f"O {tratamento} {medico_limpo}"
        texto_audio += f", localizado no bairro {bairro}, está com status de {status}. "

texto_audio += "Bom trabalho e boa sorte nas ruas de Manaus!"

In [ ]:
# 2. Converter o texto corrigido em áudio
tts = gTTS(text=texto_audio, lang='pt', slow=False)
nome_ficheiro = "rota_diaria.mp3"
tts.save(nome_ficheiro)

print("🎙️ Áudio gerado com sucesso e com os tratamentos corrigidos!")

In [ ]:
# Dados de demonstração (Mock Data) para visualização e testes no GitHub
dados_fallback = {
    'CRM': ['12345-AM', '67890-AM', '11223-AM', '44556-AM'],
    'Nome_Medico': ['Dr. Carlos Silva', 'Dra. Ana Souza', 'Dr. Roberto Lima', 'Dra. Patricia Mendes'],
    'Especialidade': ['Dermatologia', 'Pediatria', 'Cardiologia', 'Ginecologia'],
    'Target': ['Alto (A)', 'Médio (B)', 'Alto (A)', 'Baixo (C)'],
    'Bairro': ['Adrianópolis', 'Japiim', 'Centro', 'Ponta Negra'],
    'Status_Retorno': ['Urgente', 'Atenção', 'Em Dia', 'Urgente'],
    'Horario_Preferencial': ['Segundas (08h - 10h)', 'Quartas (14h - 16h)', 'Terças (10h - 12h)', 'Quintas (15h - 17h)'],
    'Linha_Produtos': ['Linha Derma', 'Linha Ped', 'Linha Cardista', 'Linha Saúde Mulher'],
    'Resumo_Ultima_Visita': ['Solicitou lâmina comparativa.', 'Boa aceitação de amostras.', 'Estoque abastecido na clínica.', 'Pendente agendamento.'],
    'Proxima_Acao': ['Entregar artigo científico.', 'Levar amostras grátis.', 'Apresentar nova apresentação.', 'Reativar contato.']
}

In [ ]:
import pandas as pd
from gtts import gTTS
import os

# 1. Simulação dos dados processados do dia
dados_hoje = [
    {"medico": "Dr. Carlos Silva", "bairro": "Adrianópolis", "prioridade": "Urgente", "linha": "Derma", "nota": "Solicitou lâmina comparativa."},
    {"medico": "Dra. Ana Souza", "bairro": "Japiim", "prioridade": "Atenção", "linha": "Pediatria", "nota": "Boa aceitação de amostras."},
]

In [ ]:
# 2. Construção do Prompt para a IA
prompt_sistema = """
Você é um assistente executivo virtual de alta performance para um representante farmacêutico em Manaus.
Sua tarefa é sintetizar as visitas do dia em um texto corrido, motivador e direto, pronto para ser lido em áudio.
Mantenha o tom profissional, elegante e breve (no máximo 4 frases).
"""

prompt_usuario = f"Gere o briefing matinal com base nestas visitas priorizadas: {dados_hoje}"

In [ ]:
# 3. Função para chamada ao Ollama Local (ou fallback de texto caso rode puro no Colab na nuvem)
def gerar_briefing_ia(prompt_sis, prompt_usr):
    try:
        import requests
        response = requests.post(
            'http://localhost:11434/api/generate',
            json={
                'model': 'llama3.2:3b',
                'prompt': f"{prompt_sis}\n\n{prompt_usr}",
                'stream': False
            },
            timeout=10
        )
        if response.status_code == 200:
            return response.json()['response']
    except Exception:
        # Fallback inteligente se o servidor Ollama não estiver rodando na porta local
        return (
            "Bom dia! Hoje o seu foco principal é no bairro Adrianópolis com o Dr. Carlos Silva, "
            "que aguarda a lâmina comparativa da Linha Derma. Em seguida, siga para o Japiim para "
            "dar atenção à Dra. Ana Souza na linha de Pediatria. Excelente trabalho e boas vendas!"
        )

In [ ]:
# 4. Execução e Geração de Áudio
texto_briefing = gerar_briefing_ia(prompt_sistema, prompt_usuario)
print(f"🤖 Briefing gerado pela IA:\n{texto_briefing}\n")

# Converte o resultado da IA em áudio
tts = gTTS(text=texto_briefing, lang='pt', slow=False)
tts.save("briefing_ia_diario.mp3")
print("🔊 Áudio sintético atualizado com sucesso via IA!")

In [ ]:
# Exporta a base (dados fictícios) para modelagem no Power BI
# sep=';' e decimal=',' evitam que o Power BI (pt-BR) leia -3.08 como -308
df_visitas.to_csv("dados_rotas_visitas.csv", index=False, encoding="utf-8-sig", sep=";", decimal=",")